<a href="https://colab.research.google.com/github/Zhengdao-Lu/Problem-Set-2-Econ_3916_PS2.ipynb/blob/main/ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6,size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    s = (base
         + 0.35 * engagement_z
         - 0.60 * had_support_ticket
         + rng.normal(0,0.75, size=n)
    )
    satisfaction = np.clip(np.round(s), 1, 5)

    return pd.DataFrame({
        'engagement_minutes': engagement_minutes,
        'engagement_z': engagement_z,
        'had_support_ticket': had_support_ticket,
        's': s,
        'satisfaction': satisfaction
    })

pre_population = simulate_population(BASE_BEFORE)
post_population = simulate_population(BASE_AFTER)

truth_summary = pd.DataFrame({
    "period": ["Before", "After"],
    "users": [len(pre_population), len(post_population)],
    "mean_latent_s": [pre_population["s"].mean(), post_population["s"].mean()],
    "true_mean_satisfaction": [pre_population["satisfaction"].mean(), post_population["satisfaction"].mean()],
})

assert len(pre_population) == len(post_population) ==50_000
assert abs(pre_population ["engagement_z"].mean())< 1e-12
assert abs(pre_population["engagement_z"].std(ddof=0) - 1) < 1e-12
assert set(pre_population["satisfaction"].unique()) <= {1, 2, 3, 4, 5}

display(truth_summary)
true_gap = post_population["satisfaction"].mean() - pre_population["satisfaction"].mean()
print(
    "CHECKPOINT truth "
    f"pre={pre_population['satisfaction'].mean():.5f} "
    f"post={post_population['satisfaction'].mean():.5f} gap={true_gap:.5f}"
)

,period,users,mean_latent_s,true_mean_satisfaction
0,Before,50000,3.3,3.291
1,After,50000,3.4,3.391


CHECKPOINT truth pre=3.29124 post=3.39092 gap=0.09968


The simulated populations contain 50000 users per period. The true mean reported satisfaction is 3.291 before and 3.391 after, which improve 0.1 points.

### Step 1.2: Impose the response mechanism (15 pts)

In [3]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""

    z = (a+b*(df["s"].to_numpy() -3.3)+0.30 * df["engagement_z"]. to_numpy()-0.25* df["had_support_ticket"].to_numpy())
    p = 1 / (1+np.exp(-z))
    return rng.random(len(df)) < p

pre_population["responded"] = respond(pre_population, a=-1.0, b=0.45)
post_population["responded"] = respond(post_population, a=-2.75, b=2.00)

pre_sample = pre_population.loc[pre_population["responded"], "satisfaction"].to_numpy()
post_sample = post_population.loc[post_population["responded"], "satisfaction"].to_numpy()

observed_gap = post_sample.mean() - pre_sample.mean()
response_summary = pd.DataFrame({
    "period": ["Before", "After"],
    "population_mean": [
        pre_population["satisfaction"].mean(),
        post_population["satisfaction"].mean(),
    ],
    "response_rate": [pre_population["responded"].mean(), post_population["responded"].mean()],
    "respondents": [len(pre_sample), len(post_sample)],
    "observed_survey_mean": [pre_sample.mean(), post_sample.mean()],
})

display(response_summary)

assert 3.55 <= pre_sample.mean() <= 3.65
assert 4.25 <= post_sample.mean() <= 4.35
assert 0.25 <= pre_population["responded"].mean() <= 0.29
assert 0.13 <= post_population["responded"].mean() <= 0.17


print(
    "CHECKPOINT response "
    f"pre_mean={pre_sample.mean():.5f} post_mean={post_sample.mean():.5f} "
    f"gap={observed_gap:.5f} pre_rate={pre_population['responded'].mean():.5f} "
    f"post_rate={post_population['responded'].mean():.5f}"
)


,period,population_mean,response_rate,respondents,observed_survey_mean
0,Before,3.291,0.275,13759,3.592
1,After,3.391,0.153,7638,4.309


CHECKPOINT response pre_mean=3.59248 post_mean=4.30937 gap=0.71689 pre_rate=0.27518 post_rate=0.15276


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

This shows a selection bias, the proability of answering is depended on latent satisfaction. The observed mean rises from 3.592 to 4.309, which shows a 0.717 point gain. THe response rate also frop to 15.3%. A larger survey will shrink sampling error around the selected group but would not make respondents representative.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [4]:
import warnings
from scipy.stats import DegenerateDataWarning


def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    bootstrap_statistics = []
    for _ in range(B):
        resample = rng.choice(x, size=len(x), replace=True)
        bootstrap_statistics.append(stat(resample))
    return np.array(bootstrap_statistics)

bootstrap_rng = np.random.default_rng(202)
interval_rows = []

for period, sample, seed in[
    ("Before", pre_sample,10),
    ("After", post_sample,20)
]:
  for statistic_name, statistic in [("Mean", np.mean), ("Median", np.median)]:
      hand_values = bootstrap_by_hand(sample, statistic, B=2000, rng=bootstrap_rng)
      hand_low, hand_high = np.percentile(hand_values, [2.5, 97.5])

      scipy_percentile = stats.bootstrap(
          (sample,), statistic, n_resamples=2000, method="percentile", random_state=seed,
      ).confidence_interval

      with warnings.catch_warnings():
        warnings.simplefilter("ignore",category=DegenerateDataWarning)
        warnings.simplefilter("ignore",category=RuntimeWarning)
        scipy_bca = stats.bootstrap(
            (sample,), statistic, n_resamples=2000, method="BCa", random_state=seed,
        ).confidence_interval

      interval_rows.append({
          "period": period,
          "statistic": statistic_name,
          "hand_low": hand_low,
          "hand_high": hand_high,
          "scipy_percentile_low": scipy_percentile.low,
          "scipy_percentile_high": scipy_percentile.high,
          "scipy_bca_low": scipy_bca[0],
          "scipy_bca_high": scipy_bca[1],
      })

interval_summary = pd.DataFrame(interval_rows)

gap_rng = np.random.default_rng(111)
bootstrap_gaps = []
for _ in range(2000):
    post_resample = gap_rng.choice(post_sample, size=len(post_sample), replace=True)
    pre_resample = gap_rng.choice(pre_sample, size=len(pre_sample), replace=True)
    bootstrap_gaps.append(post_resample.mean() - pre_resample.mean())
gap_percentile = np.percentile(bootstrap_gaps, [2.5, 97.5])
gap_bca = stats.bootstrap(
    (post_sample, pre_sample),
    lambda post, pre: post.mean() - pre.mean(),
    vectorized=False,
    paired=False,
    n_resamples=2000,
    method="BCa",
    random_state=111,
).confidence_interval

display(interval_summary)
print(
    "CHECKPOINT intervals "
    f"gap={observed_gap:.5f} percentile=[{gap_percentile[0]:.5f}, {gap_percentile[1]:.5f}] "
    f"BCa=[{gap_bca.low:.5f}, {gap_bca.high:.5f}]")

,period,statistic,hand_low,hand_high,scipy_percentile_low,scipy_percentile_high,scipy_bca_low,scipy_bca_high
0,Before,Mean,3.578,3.608,3.577,3.607,3.578,3.608
1,Before,Median,4.000,4.000,4.000,4.000,NaN,NaN
2,After,Mean,4.294,4.324,4.294,4.325,4.294,4.325
3,After,Median,4.000,4.000,4.000,4.000,NaN,NaN


CHECKPOINT intervals gap=0.71689 percentile=[0.69626, 0.73781] BCa=[0.69493, 0.73856]


*Your answer here.*

### Step 2.2: Coverage is a claim you can check (15 pts)

In [5]:
import warnings


def check_coverage(population, a, b, stat):
    rng = np.random.default_rng(2026)

    truth = stat(population["satisfaction"].to_numpy())
    n_reps = 400

    percentile_hits = 0
    bca_hits = 0
    bca_valid = 0

    for _ in range(n_reps):
        index = rng.choice(len(population), size=2000, replace=False)
        invited = population.iloc[index]


        answered = respond(invited, a, b, rng=rng)
        sample = invited.loc[answered, "satisfaction"].to_numpy()


        boot = bootstrap_by_hand(sample, stat, B=300, rng=rng)
        lo, hi = np.percentile(boot, [2.5, 97.5])

        if lo <= truth <= hi:
            percentile_hits += 1


        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            bca = stats.bootstrap(
                (sample,),
                stat,
                n_resamples=300,
                method="BCa",
                batch=100,
                random_state=rng,
            ).confidence_interval

        if np.isfinite(bca.low) and np.isfinite(bca.high):
            bca_valid += 1

            if bca.low <= truth <= bca.high:
                bca_hits += 1

    coverage = percentile_hits / n_reps
    mc_se = np.sqrt(coverage * (1 - coverage) / n_reps)

    print(f"Population truth: {truth:.3f}")
    print(f"Percentile coverage: {coverage:.3f}")
    print(f"Monte Carlo SE: {mc_se:.3f}")
    print(f"Valid BCa intervals: {bca_valid} out of {n_reps}")

    if bca_valid > 0:
        coverage = bca_hits / bca_valid
        mc_se = np.sqrt(coverage * (1 - coverage) / bca_valid)

        print(f"BCa coverage among valid intervals: {coverage:.3f}")
        print(f"Monte Carlo SE: {mc_se:.3f}")
    else:
        print("BCa coverage unavailable.")

    print()
    print("BEFORE: MEAN")
check_coverage(pre_population, -1.0, 0.45, np.mean)

print("AFTER: MEAN")
check_coverage(post_population, -2.75, 2.00, np.mean)

print("BEFORE: MEDIAN")
check_coverage(pre_population, -1.0, 0.45, np.median)

print("AFTER: MEDIAN")
check_coverage(post_population, -2.75, 2.00, np.median)

Population truth: 3.291
Percentile coverage: 0.000
Monte Carlo SE: 0.000
Valid BCa intervals: 400 out of 400
BCa coverage among valid intervals: 0.000
Monte Carlo SE: 0.000

BEFORE: MEAN
AFTER: MEAN
Population truth: 3.391
Percentile coverage: 0.000
Monte Carlo SE: 0.000
Valid BCa intervals: 400 out of 400
BCa coverage among valid intervals: 0.000
Monte Carlo SE: 0.000

BEFORE: MEAN
BEFORE: MEDIAN
Population truth: 3.000
Percentile coverage: 0.320
Monte Carlo SE: 0.023
Valid BCa intervals: 2 out of 400
BCa coverage among valid intervals: 1.000
Monte Carlo SE: 0.000

BEFORE: MEAN
AFTER: MEDIAN
Population truth: 3.000
Percentile coverage: 0.000
Monte Carlo SE: 0.000
Valid BCa intervals: 2 out of 400
BCa coverage among valid intervals: 0.000
Monte Carlo SE: 0.000

BEFORE: MEAN


Across the 400 repeated surveys, neither the percentile nor the BCa mean interval contained the true population mean in either period. The estimated converage was 0% and a monte carlo standard error of 0. The 95% upper bound after 0 successses in 400 repetitions is about 0.75%. Percentile median converage was 32.5% before, with a Monte Carlo of 2.3% point, and 0% after. Most BCa median intervals were undefined, which make their coverage not reliable. There is no oberved coverage difference between the two mean-interval methods, and differences smaller than Monte Carlo error would not establish which method is better. These intervals fail because satisfied users are likely to respond.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [6]:
test = stats.ttest_ind(post_sample, pre_sample, equal_var=False)

observed_gap = post_sample.mean() - pre_sample.mean()

n_pre = len(pre_sample)
n_post = len(post_sample)

pre_variance = pre_sample.var(ddof=1)
post_variance = post_sample.var(ddof=1)

pooled_variance = (
    (n_pre - 1) * pre_variance + (n_post - 1) * post_variance
) / (n_pre + n_post - 2)

pooled_sd = np.sqrt(pooled_variance)

cohens_d = observed_gap / pooled_sd

print(f"Before survey mean: {pre_sample.mean():.3f}")
print(f"After survey mean: {post_sample.mean():.3f}")
print(f"Observed gap: {observed_gap:.3f} points")
print(f"Welch t-statistic: {test.statistic:.3f}")
print(f"Cohen's d: {cohens_d:.3f}")

if test.pvalue == 0:
    print("p-value: less than 1e-300 (too small to display)")
else:
    print(f"p-value: {test.pvalue:.3f}")

Before survey mean: 3.592
After survey mean: 4.309
Observed gap: 0.717 points
Welch t-statistic: 66.331
Cohen's d: 0.879
p-value: less than 1e-300 (too small to display)


The response mean raise by 0.717 satisfaction points, with a Cohen's d of 0.879. The Welch test give t = 66.33 and p value less than 1e-300, it means the respondent means differ significantly. On the other hand, the redesign made satisfied users much likely to answer. And the small p-value shows that the respoindent difference is not likely to be explained bt the sampling noise, and does not remove nonresponse bias.

### Step 3.2: Power, before the fact

In [7]:
from statsmodels.stats.power import TTestIndPower

population_sd = np.sqrt(
    (
        pre_population["satisfaction"].var(ddof=1)
        + post_population["satisfaction"].var(ddof=1)
    )/2
)

planned_d = 0.10 / population_sd

power_model = TTestIndPower()

n_needed = power_model.solve_power(
    effect_size=planned_d,
    alpha=0.05,
    power=0.80,
)

print(f"Planned effect size: {planned_d:.3f}")
print(f"Responses needed per group: {int(np.ceil(n_needed))}")

actual_ratio = len(post_sample) / len(pre_sample)

d_min = power_model.solve_power(
    nobs1=len(pre_sample),
    ratio=actual_ratio,
    alpha=0.05,
    power=0.80,
)

points_min = d_min * population_sd

print(f"Before respondents: {len(pre_sample)}")
print(f"After respondents: {len(post_sample)}")
print(f"Smallest detectable d: {d_min:.3f}")
print(f"Smallest detectable points: {points_min:.3f} points")

Planned effect size: 0.113
Responses needed per group: 1237
Before respondents: 13759
After respondents: 7638
Smallest detectable d: 0.040
Smallest detectable points: 0.035 points


In [8]:
sim_rng = np.random.default_rng(3916)

true_gap = (
    post_population["satisfaction"].mean()
    - pre_population["satisfaction"].mean()
)

print(f"True population improvement: {true_gap:.3f}")

for size in [500,2000,10000,50000]:
  gaps= []

  for _ in range(250):
    pre_index = sim_rng.choice(len(pre_population), size=size, replace=False)
    post_index = sim_rng.choice(len(post_population), size=size, replace=False)

    pre_invited = pre_population.iloc[pre_index]
    post_invited = post_population.iloc[post_index]

    pre_answered = respond(pre_invited, a=-1.0, b=0.45, rng=sim_rng)
    post_answered = respond(post_invited, a=-2.75, b=2.00, rng=sim_rng)

    pre_mean = pre_invited.loc[pre_answered, "satisfaction"].mean()
    post_mean = post_invited.loc[post_answered, "satisfaction"].mean()

    gaps.append(post_mean - pre_mean)

  gaps = np.array(gaps)

  average_gap = gaps.mean()
  bias = average_gap - true_gap
  spread = gaps.std(ddof=1)

  print(f"\nUsers invited per period: {size}")
  print(f"Average estimated improvement: {average_gap:.3f}")
  print(f"Bias: {bias:.3f}")
  print(f"Survey-to-survey standard deviation:{spread:.3f}")

True population improvement: 0.100

Users invited per period: 500
Average estimated improvement: 0.710
Bias: 0.610
Survey-to-survey standard deviation:0.103

Users invited per period: 2000
Average estimated improvement: 0.707
Bias: 0.608
Survey-to-survey standard deviation:0.056

Users invited per period: 10000
Average estimated improvement: 0.710
Bias: 0.610
Survey-to-survey standard deviation:0.024

Users invited per period: 50000
Average estimated improvement: 0.709
Bias: 0.610
Survey-to-survey standard deviation:0.009


A raise of 0.1 satisfaction points corresponds to Cohen's d of approximately 0.113. A two-sided test at the 5% significace level requires 1237 completeed responses per grouo for 80% power. At actual respondent counts, the smallest detectable improvment is approximately 0.035 points. For the simulation is shows an increase invitations from 500 to 50000 which reduce the SD from 0.103 to 0.009, but the average estimated improvement stay 0.71. And the bias remian around 0.61 point. It means larger surveys reduce random variation, but doesn't fixed the selection bia.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def ipw_survey_mean(post_population):
    covariates = ["engagement_minutes", "had_support_ticket"]

    def estimate_one(data):
        # Fit response probabilities using respondents and nonrespondents.
        response_model = LogisticRegression(
            C=1e6,           # Very weak regularization
            max_iter=2000
        )
        response_model.fit(
            data[covariates],
            data["responded"]
        )

        response_probability = response_model.predict_proba(
            data[covariates]
        )[:, 1]

        # Only respondents' satisfaction enters the weighted estimate.
        respondents = data["responded"].eq(1)
        weights = 1 / response_probability[respondents]

        return np.average(
            data.loc[respondents, "satisfaction"],
            weights=weights
        )

    weighted_estimate = estimate_one(post_population)

    rng = np.random.default_rng(42)
    bootstrap_estimates = np.empty(2000)

    for b in range(2000):
        # Resample ALL population rows, including nonrespondents.
        indices = rng.integers(
            0, len(post_population), size=len(post_population)
        )
        bootstrap_sample = post_population.iloc[indices]

        # Each repetition fits a new logistic model.
        bootstrap_estimates[b] = estimate_one(bootstrap_sample)

    lower, upper = np.percentile(bootstrap_estimates, [2.5, 97.5])

    return weighted_estimate, (lower, upper)


# Corrected function call:
weighted_estimate, interval = ipw_survey_mean(post_population)

observed_respondent_mean = post_population.loc[
    post_population["responded"].eq(1), "satisfaction"
].mean()

# Available because this is a simulation with known satisfaction for all users.
true_post_redesign_mean = post_population["satisfaction"].mean()

print(f"Weighted estimate: {weighted_estimate:.3f}")
print(f"95% bootstrap interval: ({interval[0]:.3f}, {interval[1]:.3f})")
print(f"Observed respondent mean: {observed_respondent_mean:.3f}")
print(f"True post-redesign population mean: {true_post_redesign_mean:.3f}")
print(
    "Weighted estimate minus population truth: "
    f"{weighted_estimate - true_post_redesign_mean:+.3f}"
)

The correction didn't shows the true population mean. It lower the estimated from 4.309 to 4.137, but it remained 0.746 points above the true mean of 3.391. This is the expected result because response it depended on satisfaction, even after accounting for engagement and support tickets.  Weighting on those two covariates cannot fully account for the satisfied users who are more likely to answer, so selective bias remains.

### Task 4.2: The slide (5 pts)

The survey respondents reported a 0.72-point increase in satisfaction after the redesign.

It supports a difference between the users who ansered before and after. It doesn't establish an equivalent improvment among users or shows the redesign cause the satisfaction to improve. The redesigned prompt made satisfied users more likely to respond, changing the composition of respondents. The narrow interval captures sampling uncertainty but does not account for this nonresponse bias.

The data collection could replace the voluntary survey with a provavility-sample survey using a consistent process across periods.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* GPT-6.1 Sol, Oct.26

*Version 1 prompt:* [Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.


*Reply:* import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def ipw_survey_mean(
    population,
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42
):
    covariates = ["engagement_minutes", "had_support_ticket"]

    data = population.copy()
    data["had_support_ticket"] = data["had_support_ticket"].astype(int)
    data["responded"] = data["responded"].astype(int)

    def estimate_one(sample):
        X = sample[covariates]
        response = sample["responded"]

        if response.nunique() < 2 or response.sum() == 0:
            return np.nan

        # Approximate an unpenalized logistic response model.
        response_model = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=2000
        )
        response_model.fit(X, response)

        response_probability = response_model.predict_proba(X)[:, 1]

        # Only respondents contribute satisfaction scores.
        respondents = sample["responded"].eq(1)
        observed_satisfaction = sample.loc[respondents, "satisfaction"]
        inverse_probability_weight = 1 / response_probability[respondents]

        return np.average(
            observed_satisfaction,
            weights=inverse_probability_weight
        )

    weighted_estimate = estimate_one(data)

    # Resample complete population records and refit the propensity model
    # in every bootstrap iteration.
    rng = np.random.default_rng(random_state)
    bootstrap_estimates = []

    while len(bootstrap_estimates) < n_bootstrap:
        indices = rng.integers(0, len(data), size=len(data))
        bootstrap_sample = data.iloc[indices].reset_index(drop=True)

        estimate = estimate_one(bootstrap_sample)

        if np.isfinite(estimate):
            bootstrap_estimates.append(estimate)

    alpha = 1 - confidence_level
    lower, upper = np.quantile(
        bootstrap_estimates,
        [alpha / 2, 1 - alpha / 2]
    )

    return {
        "weighted_estimate": weighted_estimate,
        "bootstrap_lower": lower,
        "bootstrap_upper": upper,
        "confidence_level": confidence_level
    }


result = ipw_survey_mean(population)

print(f"IPW estimate: {result['weighted_estimate']:.3f}")
print(
    f"{result['confidence_level']:.0%} bootstrap interval: "
    f"({result['bootstrap_lower']:.3f}, "
    f"{result['bootstrap_upper']:.3f})"
)

*Version 2 prompt:* [Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] My first code failed because it called
ipw_survey_mean(population), but my notebook has no variable named
population. My post-redesign DataFrame is named post_population.
Fix the code to estimate the post-redesign population mean.

[Iterate] Use numpy, pandas and scikit-learn. Keep the function
ipw_survey_mean and use the columns engagement_minutes,
had_support_ticket, responded and satisfaction. Model response
using only engagement_minutes and had_support_ticket.

Keep the existing full-population bootstrap, which resamples all
rows and refits the logistic model in every repetition. Use
2,000 bootstrap repetitions, a 95% percentile interval and seed 42.

Show the corrected function call. Print the weighted estimate,
its interval, the observed respondent mean and the known true
post-redesign population mean. Keep the code beginner-friendly.

[Mechanism Check] Explain the assumptions needed for this weighting
method: response must be independent of satisfaction after
conditioning on the covariates, response probabilities must be
positive, and the response model must be correctly specified.
Explain why direct dependence on satisfaction violates the first
assumption.

[Evaluate] Will covariate-only weighting recover the true mean
under my response mechanism? Answer yes or no, justify your answer,
and compare the weighted estimate with the known population truth.

*Reply:* import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def ipw_survey_mean(post_population):
    covariates = ["engagement_minutes", "had_support_ticket"]

    def estimate_one(data):
        # Fit response probabilities using respondents and nonrespondents.
        response_model = LogisticRegression(
            C=1e6,           # Very weak regularization
            max_iter=2000
        )
        response_model.fit(
            data[covariates],
            data["responded"]
        )

        response_probability = response_model.predict_proba(
            data[covariates]
        )[:, 1]

        # Only respondents' satisfaction enters the weighted estimate.
        respondents = data["responded"].eq(1)
        weights = 1 / response_probability[respondents]

        return np.average(
            data.loc[respondents, "satisfaction"],
            weights=weights
        )

    weighted_estimate = estimate_one(post_population)

    rng = np.random.default_rng(42)
    bootstrap_estimates = np.empty(2000)

    for b in range(2000):
        # Resample ALL population rows, including nonrespondents.
        indices = rng.integers(
            0, len(post_population), size=len(post_population)
        )
        bootstrap_sample = post_population.iloc[indices]

        # Each repetition fits a new logistic model.
        bootstrap_estimates[b] = estimate_one(bootstrap_sample)

    lower, upper = np.percentile(bootstrap_estimates, [2.5, 97.5])

    return weighted_estimate, (lower, upper)


# Corrected function call:
weighted_estimate, interval = ipw_survey_mean(post_population)

observed_respondent_mean = post_population.loc[
    post_population["responded"].eq(1), "satisfaction"
].mean()

# Available because this is a simulation with known satisfaction for all users.
true_post_redesign_mean = post_population["satisfaction"].mean()

print(f"Weighted estimate: {weighted_estimate:.3f}")
print(f"95% bootstrap interval: ({interval[0]:.3f}, {interval[1]:.3f})")
print(f"Observed respondent mean: {observed_respondent_mean:.3f}")
print(f"True post-redesign population mean: {true_post_redesign_mean:.3f}")
print(
    "Weighted estimate minus population truth: "
    f"{weighted_estimate - true_post_redesign_mean:+.3f}"
)

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.